In [7]:
from pyspark.sql import functions as F

s = spark.table("silver_flights_clean")

a = (s.filter("other_airport_iata IS NOT NULL")
      .groupBy("other_airport_name")
      .agg(F.countDistinct("other_airport_iata").alias("n"),
           F.first("other_airport_iata", True).alias("r_iata"),
           F.first("other_airport_icao", True).alias("r_icao"),
           F.first("other_airport_timezone", True).alias("r_tz")))
ref_ap = a.filter("n = 1").drop("n")
multi = a.filter("n > 1").select("other_airport_name", F.lit(True).alias("is_multi"))
ov = spark.createDataFrame(
    [("Neom Bay Airport","NUM"), ("Kannur International Airport","CNN"),
     ("Kozhikode","CCJ"), ("Entebbe","EBB"), ("Rize","RZV")],
    ["other_airport_name", "o_iata"])

e = (s.join(ref_ap, "other_airport_name", "left")
      .join(multi, "other_airport_name", "left")
      .join(ov, "other_airport_name", "left")
      .withColumn("airport_code_source",
          F.when(F.col("other_airport_iata").isNotNull(), "original")
           .when(F.col("r_iata").isNotNull(), "reference")
           .when(F.col("o_iata").isNotNull(), "manual_override")
           .when(F.col("is_multi"), "ambiguous_city")
           .otherwise("unresolved"))
      .withColumn("other_airport_iata", F.coalesce("other_airport_iata", "r_iata", "o_iata"))
      .withColumn("other_airport_icao", F.coalesce("other_airport_icao", "r_icao"))
      .withColumn("other_airport_timezone", F.coalesce("other_airport_timezone", "r_tz"))
      .drop("n", "r_iata", "r_icao", "r_tz", "o_iata", "is_multi"))

(e.write.format("delta").mode("overwrite")
   .option("overwriteSchema", "true").saveAsTable("silver_flights"))

StatementMeta(, 3403feb6-64f0-45bf-97d5-dcb2e1300459, 9, Finished, Available, Finished, False)

In [8]:
t = spark.table("silver_flights")
print("count:", t.count())
t.groupBy("airport_code_source").count().show()

StatementMeta(, 3403feb6-64f0-45bf-97d5-dcb2e1300459, 10, Finished, Available, Finished, False)

count: 150519
+-------------------+------+
|airport_code_source| count|
+-------------------+------+
|    manual_override|   724|
|           original|146421|
|          reference|  2425|
|         unresolved|   401|
|     ambiguous_city|   548|
+-------------------+------+

